# Big Data Final Project
## Group 16 
### Group members: Mohammad Aljabrery, Nikhil Aji

Enter your `api key` after you run the following:

In [0]:
import getpass
import pandas as pd
import requests
API_KEY = getpass.getpass('Please enter your API key') 

In [0]:
BASE_URL = "https://www.googleapis.com/youtube/v3/search"
CHANNEL_ID = "UCYO_jab_esuFRV4b17AJtAw"

## Task 1

## Request to retrieve video ID, title and publish time.

In [0]:
params = {
    "key": API_KEY,
    "channelId": CHANNEL_ID,
    "part": "snippet",
    "order": "date",
    "maxResults": 10, 
}

response = requests.get(BASE_URL, params=params)
data = response.json()

In [0]:
data

### Extracting video title and publish time  from the retrieved data

In [0]:
video_info = [
    (
    item['snippet']['title'],
    item['snippet']['publishTime'],
    )
    for item in data['items']
]

In [0]:
video_info

### Extracting `video id`  from the retrieved data

In [0]:
video_ids = [
    item['id']['videoId']
    for item in data['items']
    if item['id']['kind'] == 'youtube#video'
]

In [0]:
video_ids

## Request to retrieve video view count, and comment count.

In [0]:
STATISTICS_URL = "https://www.googleapis.com/youtube/v3/videos"

stats_params = {
    "key": API_KEY,
    "id": ",".join(video_ids),
    "part": "statistics"
}

stats_response = requests.get(STATISTICS_URL, params=stats_params)
stats_data = stats_response.json()

In [0]:
stats_data

### Extracting `view count` and `comment count` of the videos from the retrieved data

In [0]:
video_stats = [
    (
    item['statistics'].get('viewCount', 0),
    item['statistics'].get('commentCount', 0)
    )
    for item in stats_data['items']
]

In [0]:
video_stats

### Combining the data

In [0]:
final_data = [
    {
        "videoId": vid,
        "title": info[0],
        "publishTime": info[1],
        "viewCount": stats[0],
        "commentCount": stats[1]
    }
    for vid, info, stats in zip(video_ids, video_info, video_stats)
]

In [0]:
final_data

### Converting the data (list) into pandas dataframe

In [0]:
df = pd.DataFrame(final_data)
df

## Converting the dataframe into a CSV `(yt_data.csv)` file

In [0]:
df.to_csv("yt_data.csv", index=False)